<a href="https://colab.research.google.com/github/scorreal25/ProyectoProgramacion/blob/main/Taller_Graficos_Publicacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller · De los datos a un dashboard publicado

**Programación — ADMI 1301**

Trabajarás con las **2.000 ventas de una cadena de papelerías en 2025**. En este taller vas a:

1. **Graficar** (6 ejercicios con `groupby`, `crosstab` y `pivot_table`).
2. **Organizar** tus gráficos en una sola figura con `subplots`: tu *dashboard*.
3. **Embellecerlo con prompts**: le pides a Gemini que reescriba el código de tu figura y pegas su respuesta aquí.
4. **Elegir tu versión favorita**, escribir tus conclusiones y **publicar tu notebook en GitHub**.

Las instrucciones completas están en la guía en Word. Aquí encuentras lo esencial de cada ejercicio y una **verificación** (`assert`) para saber si vas bien.

**Escriba los nombres de los integrantes aca.**
______________________________________________________________________  
 1.Simon Reyes   
 2.Sofia Correal   
 3.Valeria Vega  
 4. Martina Ramirez  
    


## 0. Preparación
Ejecuta la celda y sube el archivo **`ventas_papelerias_2025.csv`**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

subidos = files.upload()   # selecciona ventas_papelerias_2025.csv

df = pd.read_csv("ventas_papelerias_2025.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
df["mes"] = df["fecha"].dt.month        # 1 = enero ... 12 = diciembre
df.head()

**Conoce tus datos** antes de graficar: cuántas filas hay, qué columnas, qué valores tiene cada categoría.

In [ ]:
print(df.shape)
df.info()
for col in ["ciudad", "canal", "segmento", "categoria", "metodo_pago"]:
    print(col, "->", sorted(df[col].unique()))

---
# Parte A · Seis gráficos
En cada ejercicio: **(1)** prepara una tabla con `groupby`, `crosstab` o `pivot_table`, **(2)** guárdala en la variable indicada, **(3)** grafícala. Ponle **título** a cada gráfico.

### Ejercicio 1 · Top 5 de productos *(barras horizontales)*
**Contexto.** La gerente de compras debe decidir en qué productos concentrar el inventario del próximo año.

**Tarea.** Calcula los ingresos (`total`) por `producto`, quédate con los **5 mayores** y grafícalos en barras horizontales. Guarda la tabla en **`top_productos`**.

*Herramientas:* `groupby`, `sum`, `sort_values`, `head`, `.plot.barh()`

In [ ]:
# Ejercicio 1
top_productos = df.groupby("producto")["total"].sum().sort_values(ascending=False).head(5)

top_productos.sort_values().plot.barh()
plt.title("Top 5 productos por ingresos · 2025")
plt.xlabel("Ingresos (pesos)")
plt.ylabel("Producto")
plt.show()

In [ ]:
# ✅ Verificación
assert len(top_productos) == 5, "Debe haber exactamente 5 productos"
assert top_productos.index[0] == "Cuaderno", "El producto con más ingresos no coincide: revisa el ordenamiento"
assert abs(top_productos.iloc[0] - 20712345) < 1, "Revisa que estés sumando la columna total"
print("Ejercicio 1 correcto ✔")

### Ejercicio 2 · Ingresos mes a mes: Bogotá vs Medellín *(líneas)*
**Contexto.** La dirección regional quiere comparar la evolución de sus dos ciudades principales durante el año.

**Tarea.** **Selecciona** solo las ventas de Bogotá y Medellín (`isin`). Con una **tabla dinámica** pon los meses en las filas, las ciudades en las columnas y la suma de `total` como valores. Guárdala en **`ing_mes_ciudad`** y grafícala como líneas con marcadores.

*Herramientas:* filtrado con `isin`, `pivot_table`, `.plot.line(marker="o")`

In [ ]:
# Ejercicio 2
bog_med = df[df["ciudad"].isin(["Bogotá", "Medellín"])]

ing_mes_ciudad = bog_med.pivot_table(index="mes", columns="ciudad", values="total", aggfunc="sum")

ing_mes_ciudad.plot.line(marker="o")
plt.title("Ingresos mensuales: Bogotá vs Medellín · 2025")
plt.xlabel("Mes")
plt.ylabel("Ingresos (pesos)")
plt.show()

In [ ]:
# ✅ Verificación
assert ing_mes_ciudad.shape == (12, 2), f"Se esperaba una tabla de 12 meses x 2 ciudades, tienes {ing_mes_ciudad.shape}"
assert set(ing_mes_ciudad.columns) == {"Bogotá", "Medellín"}, "Las columnas deben ser las dos ciudades"
assert abs(ing_mes_ciudad["Bogotá"].sum() - 37847614) < 1, "El total de Bogotá no coincide"
print("Ejercicio 2 correcto ✔")

### Ejercicio 3 · ¿Por qué canal se vende más? *(gráfico de pie)*
**Contexto.** La empresa evalúa si invierte más en su tienda online o en la atención telefónica.

**Tarea.** Calcula los ingresos totales por `canal`, guárdalos en **`ing_canal`** y grafica la participación de cada canal con un gráfico de pie que muestre los **porcentajes con un decimal**.

*Herramientas:* `groupby`, `.plot.pie(autopct=...)`

In [ ]:
# Ejercicio 3
ing_canal = df.groupby("canal")["total"].sum()

ing_canal.plot.pie(autopct="%.1f%%")
plt.title("Participación de cada canal en los ingresos")
plt.ylabel("")
plt.show()

In [ ]:
# ✅ Verificación
assert abs(ing_canal.sum() - 118668012) < 1, "La suma de los canales debe ser el ingreso total"
assert abs(ing_canal["Telefónico"] / ing_canal.sum() - 0.520) < 0.01, "La participación del canal Telefónico no coincide"
print("Ejercicio 3 correcto ✔")

### Ejercicio 4 · ¿Cada segmento compra lo mismo? *(barras apiladas)*
**Contexto.** El área comercial sospecha que los clientes corporativos compran productos distintos a los minoristas.

**Tarea.** Con `pd.crosstab` cuenta las ventas por `segmento` (filas) y `categoria` (columnas), **normalizadas por fila** y expresadas en **porcentaje** (de 0 a 100). Guarda la tabla en **`tabla_seg_cat`** y grafícala en barras **apiladas**.

*Herramientas:* `pd.crosstab(..., normalize="index")`, `.plot.bar(stacked=True)`

In [ ]:
# Ejercicio 4
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100

tabla_seg_cat.plot.bar(stacked=True)
plt.title("Mezcla de categorías por segmento (% de las ventas)")
plt.xlabel("Segmento")
plt.ylabel("% de las ventas")
plt.show()

In [ ]:
# ✅ Verificación
assert tabla_seg_cat.shape == (3, 5), f"Se esperaba 3 segmentos x 5 categorías, tienes {tabla_seg_cat.shape}"
assert (abs(tabla_seg_cat.sum(axis=1) - 100) < 0.01).all(), "Cada fila debe sumar 100 (usa normalize y multiplica por 100)"
print("Ejercicio 4 correcto ✔")

### Ejercicio 5 · Productividad de los vendedores *(dispersión)*
**Contexto.** Recursos humanos revisa qué vendedores concretan más ventas y cuáles venden tickets más altos.

**Tarea.** Con `groupby` por `vendedor` y `ciudad`, calcula para cada vendedor el **número de ventas** (`n_ventas`) y el **valor promedio de sus ventas** (`ticket_prom`). Guarda la tabla en **`vend`** (con índice normal: usa `reset_index()`). Haz una dispersión (`n_ventas` en X, `ticket_prom` en Y) con un **color por ciudad** y leyenda.

*Herramientas:* `groupby().agg(nombre=("columna", "función"))`, `reset_index`, `plt.scatter` dentro de un `for`

In [ ]:
# Ejercicio 5
vend = df.groupby(["vendedor", "ciudad"]).agg(
    n_ventas=("id_venta", "count"),
    ticket_prom=("total", "mean")
).reset_index()

for ciudad in vend["ciudad"].unique():
    datos = vend[vend["ciudad"] == ciudad]
    plt.scatter(datos["n_ventas"], datos["ticket_prom"], label=ciudad)

plt.title("Vendedores: número de ventas vs ticket promedio")
plt.xlabel("Número de ventas")
plt.ylabel("Ticket promedio (pesos)")
plt.legend(title="Ciudad")
plt.show()

In [ ]:
# ✅ Verificación
assert vend.shape[0] == 12, f"Debe haber una fila por vendedor (12), tienes {vend.shape[0]}"
assert {"vendedor", "ciudad", "n_ventas", "ticket_prom"} <= set(vend.columns), "Faltan columnas: vendedor, ciudad, n_ventas, ticket_prom"
assert vend["n_ventas"].sum() == 2000, "La suma de n_ventas debe ser 2000"
print("Ejercicio 5 correcto ✔")

### Ejercicio 6 · ¿Qué tan grandes son las ventas de cada segmento? *(diagrama de cajas)*
**Contexto.** Para planear el segundo semestre, finanzas quiere ver cómo se distribuye el valor de las ventas por tipo de cliente.

**Tarea.** **Selecciona** solo las ventas del **segundo semestre** (`mes` de 7 a 12) y guárdalas en **`sem2`**. Haz un diagrama de cajas del `total` por `segmento`, **sin mostrar los valores atípicos** (`showfliers=False`).

*Herramientas:* filtro con `>=`, `.boxplot(column=..., by=...)`

In [ ]:
# Ejercicio 6
sem2 = df[df["mes"] >= 7]

sem2.boxplot(column="total", by="segmento", showfliers=False)
plt.suptitle("")
plt.title("Valor de las ventas por segmento · 2.º semestre")
plt.xlabel("Segmento")
plt.ylabel("Valor de la venta (pesos)")
plt.show()

In [ ]:
# ✅ Verificación
assert len(sem2) == 1050, f"El segundo semestre debe tener 1050 ventas, tienes {len(sem2)}"
print("Ejercicio 6 correcto ✔")

---
# Parte B · Todo en una sola figura (`subplots`)
### Ejercicio 7 · Organiza tus 6 gráficos en una cuadrícula de 2 × 3
**Contexto.** La gerente quiere ver todo en **una sola imagen** para presentarla en la junta.

**Tarea.** Crea una figura con `fig, axs = plt.subplots(2, 3, figsize=(18, 10))` y dibuja cada gráfico en su posición pasando **`ax=axs[fila, columna]`**. Agrega un título general con `fig.suptitle(...)` y termina con `plt.tight_layout()`.

**Importante:** escribe el código en la celda que empieza con `%%writefile figura_base.py` (así el programa guarda tu código en un archivo, que usarás en la Parte C). Luego ejecuta la celda siguiente (`%run -i figura_base.py`) para ver la figura.

*Pistas:* `plot.barh(ax=axs[0, 0])` · `plot.pie(...)` también acepta `ax` · para el boxplot: `boxplot(column=..., by=..., ax=...)` · para la dispersión: `axs[1, 1].scatter(...)`

In [ ]:
%%writefile figura_base.py
# Ejercicio 7: código de la figura 2x3
# Empieza con:  fig, axs = plt.subplots(2, 3, figsize=(18, 10))
fig, axs = plt.subplots(2, 3, figsize=(18, 10))

# Fila 0
top_productos.sort_values().plot.barh(ax=axs[0, 0], title="Top 5 productos por ingresos",
                                      xlabel="Ingresos (pesos)", ylabel="Producto")

ing_mes_ciudad.plot.line(ax=axs[0, 1], marker="o", title="Ingresos mensuales: Bogotá vs Medellín",
                         xlabel="Mes", ylabel="Ingresos (pesos)")

ing_canal.plot.pie(ax=axs[0, 2], autopct="%.1f%%", title="Participación de cada canal en los ingresos",
                   ylabel="")

# Fila 1
tabla_seg_cat.plot.bar(ax=axs[1, 0], stacked=True, title="Mezcla de categorías por segmento (%)",
                       xlabel="Segmento", ylabel="% de las ventas del segmento",
                       rot=0, ylim=(0, 130))
axs[1, 0].legend(title="Categoría", ncol=3, loc="upper center")

for ciudad in vend["ciudad"].unique():
    datos = vend[vend["ciudad"] == ciudad]
    axs[1, 1].scatter(datos["n_ventas"], datos["ticket_prom"], label=ciudad, s=80)
axs[1, 1].set_title("Vendedores: número de ventas vs ticket promedio")
axs[1, 1].set_xlabel("Número de ventas")
axs[1, 1].set_ylabel("Ticket promedio (pesos)")
axs[1, 1].legend(title="Ciudad")

sem2.boxplot(column="total", by="segmento", showfliers=False, ax=axs[1, 2])
axs[1, 2].set_title("Valor de las ventas por segmento · 2.º semestre")
axs[1, 2].set_xlabel("Segmento")
axs[1, 2].set_ylabel("Valor de la venta (pesos)")

fig.suptitle("Ventas de la cadena de papelerías · 2025", fontsize=18)
plt.tight_layout()


In [ ]:
%run -i figura_base.py
display(fig)

In [ ]:
# ✅ Verificación
assert len(fig.axes) >= 6, f"La figura debe tener 6 gráficos, tiene {len(fig.axes)}"
print("Ejercicio 7 correcto ✔")

---
# Parte C · Hazla ver bonita con prompts
Ahora **tú escribes el prompt** y se lo envías a Gemini (en [gemini.google.com](https://gemini.google.com) o la IA que prefieras). Gemini **reescribe el código** de tu figura para que se vea profesional, casi como un dashboard. Tú pegas ese código en el notebook y lo ejecutas.

**El ciclo, cada vez:**
1. Ejecuta la celda de herramientas: te muestra **tu código actual** y las **reglas técnicas**.
2. Escribe tu prompt (Ejercicios 8 y 9).
3. En el chat de la IA pega, en este orden: **tu prompt + tu código + las reglas**.
4. Copia **solo el código** de la respuesta (botón *copiar* del bloque) y pégalo en la celda `%%writefile figura_v1.py`. **No incluyas las líneas con \`\`\`** ni el texto que la IA escriba fuera del bloque.
5. Ejecuta la celda y luego `mostrar("v1")` para ver la figura.

### Herramientas *(ya están hechas)*
Ejecuta la celda: define la función `mostrar(nombre)` (dibuja la versión guardada en `figura_<nombre>.py`) e imprime lo que debes copiar a la IA.

In [ ]:
REGLAS_FIGURA = """

Reglas técnicas obligatorias:
- Usa solo pandas y matplotlib (y numpy si lo necesitas). No instales otras librerías.
- Los datos ya están cargados en el DataFrame `df` (columnas: id_venta, fecha, ciudad, canal, segmento, vendedor,
  producto, categoria, metodo_pago, cantidad, precio_unitario, descuento, total, mes). No cargues archivos.
- Conserva los MISMOS 6 gráficos y los mismos datos del código actual; cambia solo la presentación.
- La figura debe quedar en una variable llamada `fig` (2 filas x 3 columnas).
- NO uses plt.show() ni guardes archivos.
- Entrega solo UN bloque de código Python completo.
"""


def mostrar(nombre):
    """Dibuja la versión guardada en figura_<nombre>.py."""
    plt.close("all")
    codigo = open(f"figura_{nombre}.py", encoding="utf-8").read().replace("plt.show()", "")
    exec(codigo, globals())
    display(fig)
    plt.close(fig)


print("=== 1) COPIA ESTE CÓDIGO (tu figura actual) ===")
print(open("figura_base.py", encoding="utf-8").read())
print("\n=== 2) COPIA ESTAS REGLAS al final de tu prompt ===")
print(REGLAS_FIGURA)

### Ejercicio 8 · Tu primer prompt de estilo
Completa las secciones de `prompt_v1`. Sé **específico**: "usa una paleta de azules y naranja" funciona mejor que "hazlo bonito".

- **PÚBLICO Y OBJETIVO:** quién verá la figura y para qué.
- **PALETA Y TIPOGRAFÍA:** colores (por ciudad, por producto…), fondo, tamaños de letra.
- **TÍTULOS:** que cada gráfico tenga un título que diga la **conclusión** (por ejemplo, "Bogotá lidera las ventas"), no solo el nombre.
- **EJES Y NÚMEROS:** formatos con separador de miles o "$ 12 M", etiquetas de ejes claras.
- **LIMPIEZA VISUAL:** quitar bordes y marcos innecesarios, cuadrícula suave, valores sobre las barras.

Ejecuta la primera celda para imprimir tu prompt y cópialo a la IA (con tu código y las reglas). Luego pega el código que te devuelva en la segunda celda y muestra la figura con la tercera.

In [ ]:
prompt_v1 = """
PÚBLICO Y OBJETIVO: Actúa como un diseñador experto en dashboards con matplotlib.
La figura la verá la gerente de una cadena de papelerías en una junta, proyectada en pantalla.
Debe entender en pocos segundos qué productos, canales,
ciudades y segmentos generan más ingresos en 2025

PALETA Y TIPOGRAFÍA:Fondo gris muy claro. Usa un color fijo
por ciudad en todos los gráficos donde aparezcan: Bogotá azul oscuro, Medellín naranja,
Cali verde, Barranquilla amarillo, Bucaramanga morado.
Para canales, segmentos y categorías usa pocos colores.
Fuente sans-serif; título general en negrita tamaño 20; títulos de cada gráfico en negrita,
alineados a la izquierda, tamaño 13.

TÍTULOS: Cada gráfico debe tener un título que diga la conclusión, no solo el nombre
(por ejemplo, "Cuaderno es el producto que más ingresos genera").
Calcula las conclusiones a partir de los datos con f-strings; no escribas cifras a mano ni inventes datos.

EJES Y NÚMEROS: Muestra los pesos en millones con formato "$ 20 M" y separador de miles.
Los porcentajes llevan un decimal. Todos los ejes con etiquetas claras;
en la línea, usa nombres de meses (ene, feb, ...)

LIMPIEZA VISUAL:Quita los bordes superior y derecho de cada gráfico.
Cuadrícula delgada. Valores escritos al final de las barras.
Nada de textos superpuestos.

"""

print(prompt_v1)    # cópialo en la IA, junto con tu código y las reglas

In [ ]:
%%writefile figura_v1.py
# Pega aquí SOLO el código que te devolvió la IA (sin las líneas ```)
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

# 1. PREPARACIÓN Y CÁLCULO DE AGREGADOS A PARTIR DE `df`
# Filtrar 2025 por seguridad si hay otros años, o usar el df completo asumiendo 2025
df_2025 = df.copy()
if "fecha" in df_2025.columns:
    df_2025["fecha"] = pd.to_datetime(df_2025["fecha"])
    if (df_2025["fecha"].dt.year == 2025).any():
        df_2025 = df_2025[df_2025["fecha"].dt.year == 2025].copy()

if "mes" not in df_2025.columns and "fecha" in df_2025.columns:
    df_2025["mes"] = df_2025["fecha"].dt.month

# Datos Gráfico [0, 0]: Top 5 productos por ingresos
top_productos = (
    df_2025.groupby("producto")["total"]
    .sum()
    .nlargest(5)
    .sort_values(ascending=True)
)

# Datos Gráfico [0, 1]: Ingresos mensuales: Bogotá vs Medellín
ciudades_linea = [
    c for c in ["Bogotá", "Medellín"] if c in df_2025["ciudad"].unique()
]
if not ciudades_linea:
    ciudades_linea = df_2025["ciudad"].value_counts().nlargest(2).index.tolist()

ing_mes_ciudad = (
    df_2025[df_2025["ciudad"].isin(ciudades_linea)]
    .pivot_table(
        index="mes", columns="ciudad", values="total", aggfunc="sum", fill_value=0
    )
    .sort_index()
)
# Asegurar orden consistente de columnas
ing_mes_ciudad = ing_mes_ciudad[
    [c for c in ciudades_linea if c in ing_mes_ciudad.columns]
]

# Datos Gráfico [0, 2]: Participación de cada canal en los ingresos
ing_canal = (
    df_2025.groupby("canal")["total"].sum().sort_values(ascending=False)
)

# Datos Gráfico [1, 0]: Mezcla de categorías por segmento (%)
tabla_seg_cat = (
    pd.crosstab(
        df_2025["segmento"],
        df_2025["categoria"],
        values=df_2025["total"],
        aggfunc="sum",
        normalize="index",
    )
    * 100
)

# Datos Gráfico [1, 1]: Vendedores: número de ventas vs ticket promedio
vend = (
    df_2025.groupby(["vendedor", "ciudad"], as_index=False)
    .agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
)

# Datos Gráfico [1, 2]: Valor de las ventas por segmento · 2.º semestre
sem2 = df_2025[df_2025["mes"] >= 7].copy()
segmentos_orden = sorted(sem2["segmento"].dropna().unique())

# 2. CONFIGURACIÓN DE ESTILO, PALETA Y FORMATEADORES
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["axes.edgecolor"] = "#CCCCCC"
plt.rcParams["axes.linewidth"] = 0.8

BG_COLOR = "#F4F5F7"
CARD_COLOR = "#FFFFFF"
TEXT_DARK = "#1E293B"
TEXT_MUTED = "#475569"
GRID_COLOR = "#E2E8F0"

COLORES_CIUDAD = {
    "Bogotá": "#1B365D",       # Azul oscuro
    "Medellín": "#E66C37",     # Naranja
    "Cali": "#2E8B57",         # Verde
    "Barranquilla": "#EAB308", # Amarillo
    "Bucaramanga": "#7E22CE",  # Morado
}

PALETA_SOBRIA = ["#2B4C7E", "#567EBB", "#8FAADC", "#A3B1C6", "#CBD5E1"]
PALETA_CANALES = ["#1B365D", "#3B6998", "#6B9AC4", "#97B8D6", "#C3D7EA"]

def fmt_millones(val):
    mill = val / 1e6
    if abs(mill - round(mill)) < 0.05:
        return f"$ {int(round(mill)):,} M".replace(",", ".")
    return f"$ {mill:,.1f} M".replace(",", "X").replace(".", ",").replace("X", ".")

def fmt_eje_millones(x, pos):
    mill = x / 1e6
    if x == 0:
        return "$ 0 M"
    if abs(mill - round(mill)) < 1e-6:
        return f"$ {int(round(mill)):,} M".replace(",", ".")
    return f"$ {mill:,.1f} M".replace(",", "X").replace(".", ",").replace("X", ".")

def fmt_eje_ticket(x, pos):
    if x >= 1e6:
        return fmt_eje_millones(x, pos)
    if x >= 1e3:
        return f"$ {int(round(x / 1e3)):,} k".replace(",", ".")
    return f"$ {int(round(x)):,}".replace(",", ".")

def estilizar_eje(ax, titulo, grid_axis="y"):
    ax.set_facecolor(CARD_COLOR)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#CBD5E1")
    ax.spines["bottom"].set_color("#CBD5E1")
    ax.tick_params(colors=TEXT_MUTED, labelsize=9.5)
    ax.xaxis.label.set_color(TEXT_DARK)
    ax.yaxis.label.set_color(TEXT_DARK)
    ax.xaxis.label.set_fontsize(10)
    ax.yaxis.label.set_fontsize(10)
    if grid_axis:
        ax.grid(True, axis=grid_axis, linestyle="--", linewidth=0.6, color=GRID_COLOR, alpha=0.9)
        ax.set_axisbelow(True)
    ax.set_title(
        titulo,
        fontsize=13,
        fontweight="bold",
        loc="left",
        color=TEXT_DARK,
        pad=12,
    )

# 3. CREACIÓN DEL LIENZO (2 FILAS x 3 COLUMNAS)
fig, axs = plt.subplots(2, 3, figsize=(18, 10), facecolor=BG_COLOR)

# --- GRÁFICO [0, 0]: TOP 5 PRODUCTOS ---
ax = axs[0, 0]
prod_lider = top_productos.index[-1]
val_prod_lider = top_productos.iloc[-1]
colores_prod = ["#94A3B8"] * (len(top_productos) - 1) + ["#1B365D"]

bars = ax.barh(top_productos.index, top_productos.values, color=colores_prod, height=0.62)
max_x_prod = top_productos.max()
ax.set_xlim(0, max_x_prod * 1.24)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_eje_millones))
ax.set_xlabel("Ingresos totales")
ax.set_ylabel("Producto")

for bar in bars:
    w = bar.get_width()
    ax.text(
        w + max_x_prod * 0.02,
        bar.get_y() + bar.get_height() / 2,
        fmt_millones(w),
        va="center",
        ha="left",
        fontsize=9.5,
        fontweight="bold" if w == val_prod_lider else "normal",
        color=TEXT_DARK,
    )

titulo_00 = f"{prod_lider} lidera los ingresos con {fmt_millones(val_prod_lider)}"
estilizar_eje(ax, titulo_00, grid_axis="x")

# --- GRÁFICO [0, 1]: INGRESOS MENSUALES BOGOTÁ VS MEDELLÍN ---
ax = axs[0, 1]
meses_nombres = {
    1: "ene", 2: "feb", 3: "mar", 4: "abr", 5: "may", 6: "jun",
    7: "jul", 8: "ago", 9: "sep", 10: "oct", 11: "nov", 12: "dic",
}
x_idx = np.arange(len(ing_mes_ciudad.index))
etiquetas_mes = [meses_nombres.get(int(m), str(m)) for m in ing_mes_ciudad.index]

for ciudad in ing_mes_ciudad.columns:
    color_c = COLORES_CIUDAD.get(ciudad, "#334155")
    ax.plot(
        x_idx,
        ing_mes_ciudad[ciudad].values,
        marker="o",
        markersize=5.5,
        linewidth=2.3,
        color=color_c,
        label=ciudad,
    )

ax.set_xticks(x_idx)
ax.set_xticklabels(etiquetas_mes)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(fmt_eje_millones))
ax.set_ylim(0, ing_mes_ciudad.values.max() * 1.18)
ax.set_xlabel("Mes")
ax.set_ylabel("Ingresos mensuales")
ax.legend(frameon=False, loc="upper left", ncol=2, fontsize=9.5)

totales_ciudades_linea = ing_mes_ciudad.sum()
ciudad_mayor = totales_ciudades_linea.idxmax()
ciudad_menor = totales_ciudades_linea.idxmin()
dif_ciudades = totales_ciudades_linea[ciudad_mayor] - totales_ciudades_linea[ciudad_menor]
titulo_01 = (
    f"{ciudad_mayor} supera a {ciudad_menor} por {fmt_millones(dif_ciudades)}"
    if len(ing_mes_ciudad.columns) > 1
    else f"{ciudad_mayor} alcanza {fmt_millones(totales_ciudades_linea[ciudad_mayor])} en el año"
)
estilizar_eje(ax, titulo_01, grid_axis="y")

# --- GRÁFICO [0, 2]: PARTICIPACIÓN POR CANAL ---
ax = axs[0, 2]
ax.set_facecolor(CARD_COLOR)
canal_top = ing_canal.index[0]
pct_canal_top = (ing_canal.iloc[0] / ing_canal.sum()) * 100
explode_vals = [0.04 if i == 0 else 0.0 for i in range(len(ing_canal))]

wedges, texts, autotexts = ax.pie(
    ing_canal.values,
    labels=ing_canal.index,
    autopct="%.1f%%",
    startangle=90,
    counterclock=False,
    colors=PALETA_CANALES[: len(ing_canal)],
    explode=explode_vals,
    pctdistance=0.68,
    labeldistance=1.12,
    wedgeprops={"edgecolor": CARD_COLOR, "linewidth": 2},
    textprops={"fontsize": 10, "color": TEXT_DARK},
)
for i, autotext in enumerate(autotexts):
    autotext.set_color("white" if i < 2 else TEXT_DARK)
    autotext.set_fontweight("bold")
    autotext.set_fontsize(9.5)

titulo_02 = f"Canal {canal_top} concentra el {pct_canal_top:.1f}% de las ventas"
ax.set_title(titulo_02, fontsize=13, fontweight="bold", loc="left", color=TEXT_DARK, pad=12)

# --- GRÁFICO [1, 0]: MEZCLA DE CATEGORÍAS POR SEGMENTO (%) ---
ax = axs[1, 0]
categorias = tabla_seg_cat.columns.tolist()
colores_cat = PALETA_SOBRIA[: len(categorias)]
bottom_vals = np.zeros(len(tabla_seg_cat))
x_seg = np.arange(len(tabla_seg_cat.index))

for idx_c, cat in enumerate(categorias):
    vals = tabla_seg_cat[cat].values
    ax.bar(
        x_seg,
        vals,
        bottom=bottom_vals,
        width=0.55,
        color=colores_cat[idx_c % len(colores_cat)],
        label=cat,
        edgecolor=CARD_COLOR,
        linewidth=1,
    )
    for j, v in enumerate(vals):
        if v >= 10:
            ax.text(
                x_seg[j],
                bottom_vals[j] + v / 2,
                f"{v:.1f}%",
                ha="center",
                va="center",
                fontsize=8.8,
                fontweight="bold",
                color="white" if idx_c < 2 else TEXT_DARK,
            )
    bottom_vals += vals

ax.set_xticks(x_seg)
ax.set_xticklabels(tabla_seg_cat.index)
ax.set_ylim(0, 122)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.set_xlabel("Segmento")
ax.set_ylabel("Participación en ventas (%)")
ax.legend(
    title="Categoría",
    ncol=min(3, len(categorias)),
    loc="upper center",
    frameon=False,
    fontsize=9,
    title_fontsize=9.5,
)

# Encontrar par segmento-categoría dominante
seg_dom, cat_dom = tabla_seg_cat.stack().idxmax()
pct_dom = tabla_seg_cat.loc[seg_dom, cat_dom]
titulo_10 = f"{cat_dom} representa {pct_dom:.1f}% en segmento {seg_dom}"
estilizar_eje(ax, titulo_10, grid_axis="y")

# --- GRÁFICO [1, 1]: DISPERSIÓN VENDEDORES ---
ax = axs[1, 1]
ciudades_ordenadas = [c for c in COLORES_CIUDAD if c in vend["ciudad"].unique()] + [
    c for c in vend["ciudad"].unique() if c not in COLORES_CIUDAD
]

for ciudad in ciudades_ordenadas:
    datos = vend[vend["ciudad"] == ciudad]
    ax.scatter(
        datos["n_ventas"],
        datos["ticket_prom"],
        label=ciudad,
        color=COLORES_CIUDAD.get(ciudad, "#64748B"),
        s=75,
        alpha=0.88,
        edgecolors="white",
        linewidth=0.8,
    )

ax.yaxis.set_major_formatter(ticker.FuncFormatter(fmt_eje_ticket))
ax.set_ylim(0, vend["ticket_prom"].max() * 1.22)
ax.set_xlabel("Número de ventas por vendedor")
ax.set_ylabel("Ticket promedio")
ax.legend(
    title="Ciudad",
    frameon=False,
    loc="upper right",
    ncol=2,
    fontsize=8.5,
    title_fontsize=9,
)

ciudad_mayor_ticket = vend.groupby("ciudad")["ticket_prom"].mean().idxmax()
ticket_top_ciudad = vend.groupby("ciudad")["ticket_prom"].mean().max()
titulo_11 = f"{ciudad_mayor_ticket} logra el mayor ticket medio ({fmt_eje_ticket(ticket_top_ciudad, 0)})"
estilizar_eje(ax, titulo_11, grid_axis="both")

# --- GRÁFICO [1, 2]: BOXPLOT 2.º SEMESTRE POR SEGMENTO ---
ax = axs[1, 2]
datos_box = [sem2[sem2["segmento"] == s]["total"].dropna().values for s in segmentos_orden]
bp = ax.boxplot(
    datos_box,
    labels=segmentos_orden,
    showfliers=False,
    patch_artist=True,
    widths=0.45,
    medianprops={"color": "#E66C37", "linewidth": 2.2},
    boxprops={"facecolor": "#CBD5E1", "edgecolor": "#334155", "linewidth": 1.1},
    whiskerprops={"color": "#334155", "linewidth": 1.1},
    capprops={"color": "#334155", "linewidth": 1.1},
)

medianas_sem2 = sem2.groupby("segmento")["total"].median()
seg_med_top = medianas_sem2.idxmax()
val_med_top = medianas_sem2.max()

# Destacar la caja con mayor mediana y etiquetar medianas limpiamente
for idx_s, seg_name in enumerate(segmentos_orden):
    if seg_name == seg_med_top:
        bp["boxes"][idx_s].set_facecolor("#1B365D")
        bp["boxes"][idx_s].set_alpha(0.85)
    med_val = medianas_sem2[seg_name]
    q3_val = np.percentile(datos_box[idx_s], 75) if len(datos_box[idx_s]) > 0 else med_val
    ax.text(
        idx_s + 1,
        q3_val * 1.04,
        f"Med: {fmt_eje_ticket(med_val, 0)}",
        ha="center",
        va="bottom",
        fontsize=8.8,
        fontweight="bold" if seg_name == seg_med_top else "normal",
        color=TEXT_DARK,
    )

max_whisker = max(
    (np.percentile(d, 75) + 1.5 * (np.percentile(d, 75) - np.percentile(d, 25)))
    for d in datos_box
    if len(d) > 0
)
ax.set_ylim(0, max_whisker * 1.18)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(fmt_eje_ticket))
ax.set_xlabel("Segmento")
ax.set_ylabel("Valor por venta (2.º sem.)")

titulo_12 = f"Segmento {seg_med_top} lidera la mediana semestral ({fmt_eje_ticket(val_med_top, 0)})"
estilizar_eje(ax, titulo_12, grid_axis="y")

# 4. TÍTULO GENERAL Y AJUSTE FINAL DE ESPACIOS
ingreso_total_2025 = df_2025["total"].sum()
fig.suptitle(
    f"Desempeño comercial de la cadena de papelerías · 2025 (Total: {fmt_millones(ingreso_total_2025)})",
    fontsize=20,
    fontweight="bold",
    color=TEXT_DARK,
    x=0.04,
    y=0.97,
    ha="left",
)

plt.tight_layout(rect=[0.02, 0.02, 0.99, 0.92], h_pad=2.8, w_pad=2.4)

In [ ]:
mostrar("v1")

### Ejercicio 9 · Itera: mejora tu prompt
Mira la figura que obtuviste: ¿qué no te gusta? Agrega una sección **AJUSTES** con **al menos dos cambios concretos** (otra paleta, letras más grandes, una dona en lugar del pie, anotar el valor máximo del gráfico de líneas…). Es la **iteración de prompts** que vimos en clase. Repite el ciclo: copia `prompt_v2` a la IA (con tu código base y las reglas), pega el código en `figura_v2.py` y muestra la figura.

In [ ]:
prompt_v2 = prompt_v1 + """

AJUSTES: Cambia el color de énfasis (las barras y porciones principales, que
ahora son azul oscuro) a verde, y deja el resto en gris claro.
Mantén los colores fijos por ciudad.Haz las letras de las leyendas y de los ejes
 un poco más grandes para que se lean proyectadas
"""

print(prompt_v2)

In [ ]:
%%writefile figura_v2.py
# Pega aquí el código que te devolvió la IA para la versión 2
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

# 1. DATOS Y AGREGADOS
df_2025 = df.copy()
if "fecha" in df_2025.columns:
    df_2025["fecha"] = pd.to_datetime(df_2025["fecha"])
    if (df_2025["fecha"].dt.year == 2025).any():
        df_2025 = df_2025[df_2025["fecha"].dt.year == 2025].copy()
if "mes" not in df_2025.columns and "fecha" in df_2025.columns:
    df_2025["mes"] = df_2025["fecha"].dt.month

top_productos = df_2025.groupby("producto")["total"].sum().nlargest(5).sort_values()

ciudades_linea = [c for c in ["Bogotá", "Medellín"] if c in df_2025["ciudad"].unique()]
if not ciudades_linea:
    ciudades_linea = df_2025["ciudad"].value_counts().nlargest(2).index.tolist()
ing_mes_ciudad = (
    df_2025[df_2025["ciudad"].isin(ciudades_linea)]
    .pivot_table(index="mes", columns="ciudad", values="total", aggfunc="sum", fill_value=0)
    .reindex(columns=ciudades_linea)
    .sort_index()
)

ing_canal = df_2025.groupby("canal")["total"].sum().sort_values(ascending=False)
tabla_seg_cat = pd.crosstab(df_2025["segmento"], df_2025["categoria"], values=df_2025["total"], aggfunc="sum", normalize="index") * 100
vend = df_2025.groupby(["vendedor", "ciudad"], as_index=False).agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
sem2 = df_2025[df_2025["mes"] >= 7]
segmentos_orden = sorted(sem2["segmento"].dropna().unique())

# 2. ESTILO Y FORMATEADORES
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
BG, CARD, DARK, MUTED = "#F4F5F7", "#FFFFFF", "#1E293B", "#475569"
VERDE = "#2E8B57"
GRISES = ["#CBD5E1", "#DCE3EC", "#E8EDF3", "#F1F5F9"]
COLOR_CIUDAD = {"Bogotá": "#1B365D", "Medellín": "#E66C37", "Cali": "#2E8B57", "Barranquilla": "#EAB308", "Bucaramanga": "#7E22CE"}
MESES = {1: "ene", 2: "feb", 3: "mar", 4: "abr", 5: "may", 6: "jun", 7: "jul", 8: "ago", 9: "sep", 10: "oct", 11: "nov", 12: "dic"}

def fmt_m(v):
    m = v / 1e6
    s = f"{int(round(m)):,}" if abs(m - round(m)) < 0.05 else f"{m:,.1f}"
    return "$ " + s.replace(",", "X").replace(".", ",").replace("X", ".") + " M"

def fmt_t(x, _=None):
    return fmt_m(x) if x >= 1e6 else (f"$ {int(round(x / 1e3)):,} k".replace(",", ".") if x >= 1e3 else f"$ {int(round(x)):,}".replace(",", "."))

def estilo(ax, titulo, grid="y"):
    ax.set_facecolor(CARD)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#CBD5E1")
    ax.tick_params(colors=MUTED, labelsize=11)
    ax.xaxis.label.set(color=DARK, fontsize=11.5)
    ax.yaxis.label.set(color=DARK, fontsize=11.5)
    if grid:
        ax.grid(True, axis=grid, linestyle="--", linewidth=0.6, color="#E2E8F0")
        ax.set_axisbelow(True)
    ax.set_title(titulo, fontsize=13, fontweight="bold", loc="left", color=DARK, pad=12)

# 3. FIGURA Y SUBPLOTS
fig, axs = plt.subplots(2, 3, figsize=(18, 10), facecolor=BG)

# [0, 0] Top 5 productos
ax = axs[0, 0]
bars = ax.barh(top_productos.index, top_productos.values, color=["#CBD5E1"] * (len(top_productos) - 1) + [VERDE], height=0.62)
ax.set_xlim(0, top_productos.max() * 1.27)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: fmt_m(x)))
ax.set(xlabel="Ingresos totales", ylabel="Producto")
for i, b in enumerate(bars):
    ax.text(b.get_width() + top_productos.max() * 0.02, b.get_y() + b.get_height() / 2, fmt_m(b.get_width()),
            va="center", fontsize=10, fontweight="bold" if i == len(bars) - 1 else "normal", color=DARK)
estilo(ax, f"{top_productos.index[-1]} lidera los ingresos con {fmt_m(top_productos.iloc[-1])}", "x")

# [0, 1] Línea mensual por ciudad
ax = axs[0, 1]
x_idx = np.arange(len(ing_mes_ciudad))
etq_mes = [MESES.get(int(m), str(m)) for m in ing_mes_ciudad.index]
for i, ciudad in enumerate(ing_mes_ciudad.columns):
    c = COLOR_CIUDAD.get(ciudad, "#334155")
    vals = ing_mes_ciudad[ciudad].values
    ax.plot(x_idx, vals, marker="o", markersize=5.5, lw=2.3, color=c, label=ciudad)
    p_max = int(np.argmax(vals))
    ox = 14 if p_max <= 1 else (-14 if p_max >= len(x_idx) - 2 else 0)
    ax.annotate(f"{etq_mes[p_max].capitalize()}: {fmt_m(vals[p_max])}", xy=(p_max, vals[p_max]),
                xytext=(ox, 20 if i == 0 else 18), textcoords="offset points",
                ha="left" if ox > 0 else ("right" if ox < 0 else "center"), fontsize=9.5, fontweight="bold", color=c,
                bbox=dict(boxstyle="round,pad=0.22", fc=CARD, ec=c, lw=0.9),
                arrowprops=dict(arrowstyle="->", color=c, lw=1.2, shrinkB=4))
ax.set_xticks(x_idx)
ax.set_xticklabels(etq_mes)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: fmt_m(x)))
ax.set_ylim(0, ing_mes_ciudad.values.max() * 1.30)
ax.set(xlabel="Mes", ylabel="Ingresos mensuales")
ax.legend(frameon=False, loc="upper left", ncol=2, fontsize=12)
tot_c = ing_mes_ciudad.sum()
t01 = f"{tot_c.idxmax()} supera a {tot_c.idxmin()} por {fmt_m(tot_c.max() - tot_c.min())}" if len(tot_c) > 1 else f"{tot_c.idxmax()} alcanza {fmt_m(tot_c.max())}"
estilo(ax, t01, "y")

# [0, 2] Dona de canales
ax = axs[0, 2]
ax.set_facecolor(CARD)
_, _, autotexts = ax.pie(
    ing_canal.values, labels=ing_canal.index, autopct="%.1f%%", startangle=90, counterclock=False,
    colors=[VERDE] + [GRISES[i % len(GRISES)] for i in range(len(ing_canal) - 1)],
    explode=[0.035] + [0.0] * (len(ing_canal) - 1), pctdistance=0.73, labeldistance=1.10,
    wedgeprops=dict(width=0.46, edgecolor=CARD, lw=2.2), textprops=dict(fontsize=11.5, color=DARK)
)
for i, at in enumerate(autotexts):
    at.set(color="white" if i == 0 else DARK, fontweight="bold", fontsize=10.5)
ax.text(0, 0.05, "Total 2025", ha="center", va="center", fontsize=10, color=MUTED)
ax.text(0, -0.11, fmt_m(ing_canal.sum()), ha="center", va="center", fontsize=12, fontweight="bold", color=DARK)
ax.set_title(f"Canal {ing_canal.index[0]} concentra el {ing_canal.iloc[0] / ing_canal.sum() * 100:.1f}% de las ventas",
             fontsize=13, fontweight="bold", loc="left", color=DARK, pad=12)

# [1, 0] Barras apiladas categoría por segmento
ax = axs[1, 0]
cats = tabla_seg_cat.columns.tolist()
cat_top = tabla_seg_cat.mean().idxmax()
bottom = np.zeros(len(tabla_seg_cat))
x_seg = np.arange(len(tabla_seg_cat))
g_idx = 0
for cat in cats:
    es_top = cat == cat_top
    color = VERDE if es_top else GRISES[g_idx % len(GRISES)]
    g_idx += not es_top
    vals = tabla_seg_cat[cat].values
    ax.bar(x_seg, vals, bottom=bottom, width=0.55, color=color, label=cat, edgecolor=CARD, lw=1.2)
    for j, v in enumerate(vals):
        if v >= 10:
            ax.text(x_seg[j], bottom[j] + v / 2, f"{v:.1f}%", ha="center", va="center",
                    fontsize=9.5, fontweight="bold", color="white" if es_top else DARK)
    bottom += vals
ax.set_xticks(x_seg)
ax.set_xticklabels(tabla_seg_cat.index)
ax.set_ylim(0, 128)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.set(xlabel="Segmento", ylabel="Participación en ventas (%)")
ax.legend(title="Categoría", ncol=min(3, len(cats)), loc="upper center", frameon=False, fontsize=11.5, title_fontsize=12)
seg_dom, cat_dom = tabla_seg_cat.stack().idxmax()
estilo(ax, f"{cat_dom} representa {tabla_seg_cat.loc[seg_dom, cat_dom]:.1f}% en segmento {seg_dom}", "y")

# [1, 1] Dispersión de vendedores
ax = axs[1, 1]
ciudades_ord = [c for c in COLOR_CIUDAD if c in vend["ciudad"].unique()] + [c for c in vend["ciudad"].unique() if c not in COLOR_CIUDAD]
for ciudad in ciudades_ord:
    d = vend[vend["ciudad"] == ciudad]
    ax.scatter(d["n_ventas"], d["ticket_prom"], label=ciudad, color=COLOR_CIUDAD.get(ciudad, "#64748B"),
               s=85, alpha=0.88, edgecolors="white", lw=0.8)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(fmt_t))
ax.set_ylim(0, vend["ticket_prom"].max() * 1.28)
ax.set(xlabel="Número de ventas por vendedor", ylabel="Ticket promedio")
ax.legend(title="Ciudad", frameon=False, loc="upper right", ncol=2, fontsize=11, title_fontsize=11.5, markerscale=1.25)
t_ciudad = vend.groupby("ciudad")["ticket_prom"].mean()
estilo(ax, f"{t_ciudad.idxmax()} logra el mayor ticket medio ({fmt_t(t_ciudad.max())})", "both")

# [1, 2] Boxplot 2.º semestre
ax = axs[1, 2]
datos_box = [sem2[sem2["segmento"] == s]["total"].dropna().values for s in segmentos_orden]
bp = ax.boxplot(datos_box, labels=segmentos_orden, showfliers=False, patch_artist=True, widths=0.45,
                medianprops=dict(color=DARK, lw=2.2), boxprops=dict(facecolor="#E2E8F0", edgecolor=MUTED, lw=1.1),
                whiskerprops=dict(color=MUTED, lw=1.1), capprops=dict(color=MUTED, lw=1.1))
meds = sem2.groupby("segmento")["total"].median()
seg_top = meds.idxmax()
for i, s in enumerate(segmentos_orden):
    if s == seg_top:
        bp["boxes"][i].set(facecolor=VERDE, alpha=0.88)
        bp["medians"][i].set_color("white")
    q3 = np.percentile(datos_box[i], 75) if len(datos_box[i]) else meds[s]
    ax.text(i + 1, q3 * 1.04, f"Med: {fmt_t(meds[s])}", ha="center", fontsize=9.5,
            fontweight="bold" if s == seg_top else "normal", color=DARK)
max_w = max((np.percentile(d, 75) + 1.5 * (np.percentile(d, 75) - np.percentile(d, 25))) for d in datos_box if len(d))
ax.set_ylim(0, max_w * 1.20)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(fmt_t))
ax.set(xlabel="Segmento", ylabel="Valor por venta (2.º sem.)")
estilo(ax, f"Segmento {seg_top} lidera la mediana semestral ({fmt_t(meds.max())})", "y")

fig.suptitle(f"Desempeño comercial de la cadena de papelerías · 2025 (Total: {fmt_m(df_2025['total'].sum())})",
             fontsize=20, fontweight="bold", color=DARK, x=0.04, y=0.97, ha="left")
plt.tight_layout(rect=[0.02, 0.02, 0.99, 0.92], h_pad=2.8, w_pad=2.4)

In [ ]:
# Definimos 'mostrar' localmente por si se reinició el entorno
def mostrar(nombre):
    import matplotlib.pyplot as plt
    plt.close("all")
    codigo = open(f"figura_{nombre}.py", encoding="utf-8").read().replace("plt.show()", "")
    exec(codigo, globals())
    display(fig)
    plt.close(fig)

mostrar("v2")

### Ejercicio 10 *(opcional)* · Una tercera versión
Si quieres seguir mejorando, crea `prompt_v3` (parte de `prompt_v2`), repite el ciclo con `figura_v3.py` y muestra `mostrar("v3")`.

In [ ]:
# prompt_v3 = prompt_v2 + """
#
# AJUSTES FINALES: ...
# """
# print(prompt_v3)

In [ ]:
# ✅ Verificación
import os
assert os.path.exists("figura_v1.py") and os.path.exists("figura_v2.py"), "Faltan figura_v1.py y figura_v2.py: pega el código de cada versión"
assert "ESCRIBE AQUÍ" not in prompt_v1 and prompt_v2 != prompt_v1 and "ESCRIBE AQUÍ" not in prompt_v2, "Completa tus prompts v1 y v2"
print("Parte C correcta ✔")

---
# Parte D · Elige tu favorita y publícala
### Ejercicio 11 · Tu dashboard final y tus conclusiones
Mira tus versiones y **elige la que más te guste**. Luego escribe **3 hallazgos**: frases cortas con **cifras reales** sacadas de tus tablas. Se mostrarán debajo de tu dashboard.

*Ejemplo de hallazgo:* "Bogotá genera el 32 % de los ingresos de la cadena."

In [ ]:
figura_elegida = "v2"      # cambia por "v1", "v2" o "v3"

hallazgos = [
    "El canal Telofónico concentra el 51.8 %  de las ventas totales de la cadena ($ 118,7 M).",
    "Cuaderno es el producto lider en ingresos alcanzando $20,7 M.",
    " La categoria Papel representa el 52,9 % de la participacion en ventas del segmento Corporativo. ",
]

In [ ]:
# ✅ Verificación
import os
assert os.path.exists(f"figura_{figura_elegida}.py"), f"No existe figura_{figura_elegida}.py: revisa figura_elegida"
assert len(hallazgos) == 3 and all(len(h) > 25 and "ESCRIBE AQUÍ" not in h for h in hallazgos), "Escribe tus 3 hallazgos con cifras"
print("Ejercicio 11 correcto ✔")

In [ ]:
import os

codigo_v2 = """
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

# 1. DATOS Y AGREGADOS
df_2025 = df.copy()
if "fecha" in df_2025.columns:
    df_2025["fecha"] = pd.to_datetime(df_2025["fecha"])
    if (df_2025["fecha"].dt.year == 2025).any():
        df_2025 = df_2025[df_2025["fecha"].dt.year == 2025].copy()
if "mes" not in df_2025.columns and "fecha" in df_2025.columns:
    df_2025["mes"] = df_2025["fecha"].dt.month

top_productos = df_2025.groupby("producto")["total"].sum().nlargest(5).sort_values()

ciudades_linea = [c for c in ["Bogotá", "Medellín"] if c in df_2025["ciudad"].unique()]
if not ciudades_linea:
    ciudades_linea = df_2025["ciudad"].value_counts().nlargest(2).index.tolist()
ing_mes_ciudad = (
    df_2025[df_2025["ciudad"].isin(ciudades_linea)]
    .pivot_table(index="mes", columns="ciudad", values="total", aggfunc="sum", fill_value=0)
    .reindex(columns=ciudades_linea)
    .sort_index()
)

ing_canal = df_2025.groupby("canal")["total"].sum().sort_values(ascending=False)
tabla_seg_cat = pd.crosstab(df_2025["segmento"], df_2025["categoria"], values=df_2025["total"], aggfunc="sum", normalize="index") * 100
vend = df_2025.groupby(["vendedor", "ciudad"], as_index=False).agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
sem2 = df_2025[df_2025["mes"] >= 7]
segmentos_orden = sorted(sem2["segmento"].dropna().unique())

# 2. ESTILO Y FORMATEADORES
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
BG, CARD, DARK, MUTED = "#F4F5F7", "#FFFFFF", "#1E293B", "#475569"
VERDE = "#2E8B57"
GRISES = ["#CBD5E1", "#DCE3EC", "#E8EDF3", "#F1F5F9"]
COLOR_CIUDAD = {"Bogotá": "#1B365D", "Medellín": "#E66C37", "Cali": "#2E8B57", "Barranquilla": "#EAB308", "Bucaramanga": "#7E22CE"}
MESES = {1: "ene", 2: "feb", 3: "mar", 4: "abr", 5: "may", 6: "jun", 7: "jul", 8: "ago", 9: "sep", 10: "oct", 11: "nov", 12: "dic"}

def fmt_m(v):
    m = v / 1e6
    s = f"{int(round(m)):,}" if abs(m - round(m)) < 0.05 else f"{m:,.1f}"
    return "$ " + s.replace(",", "X").replace(".", ",").replace("X", ".") + " M"

def fmt_t(x, _=None):
    return fmt_m(x) if x >= 1e6 else (f"$ {int(round(x / 1e3)):,} k".replace(",", ".") if x >= 1e3 else f"$ {int(round(x)):,}".replace(",", "."))

def estilo(ax, titulo, grid="y"):
    ax.set_facecolor(CARD)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#CBD5E1")
    ax.tick_params(colors=MUTED, labelsize=11)
    ax.xaxis.label.set(color=DARK, fontsize=11.5)
    ax.yaxis.label.set(color=DARK, fontsize=11.5)
    if grid:
        ax.grid(True, axis=grid, linestyle="--", linewidth=0.6, color="#E2E8F0")
        ax.set_axisbelow(True)
    ax.set_title(titulo, fontsize=13, fontweight="bold", loc="left", color=DARK, pad=12)

# 3. FIGURA Y SUBPLOTS
fig, axs = plt.subplots(2, 3, figsize=(18, 10), facecolor=BG)

# [0, 0] Top 5 productos
ax = axs[0, 0]
bars = ax.barh(top_productos.index, top_productos.values, color=["#CBD5E1"] * (len(top_productos) - 1) + [VERDE], height=0.62)
ax.set_xlim(0, top_productos.max() * 1.27)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: fmt_m(x)))
ax.set(xlabel="Ingresos totales", ylabel="Producto")
for i, b in enumerate(bars):
    ax.text(b.get_width() + top_productos.max() * 0.02, b.get_y() + b.get_height() / 2, fmt_m(b.get_width()),
            va="center", fontsize=10, fontweight="bold" if i == len(bars) - 1 else "normal", color=DARK)
estilo(ax, f"{top_productos.index[-1]} lidera los ingresos con {fmt_m(top_productos.iloc[-1])}", "x")

# [0, 1] Línea mensual por ciudad
ax = axs[0, 1]
x_idx = np.arange(len(ing_mes_ciudad))
etq_mes = [MESES.get(int(m), str(m)) for m in ing_mes_ciudad.index]
for i, ciudad in enumerate(ing_mes_ciudad.columns):
    c = COLOR_CIUDAD.get(ciudad, "#334155")
    vals = ing_mes_ciudad[ciudad].values
    ax.plot(x_idx, vals, marker="o", markersize=5.5, lw=2.3, color=c, label=ciudad)
    p_max = int(np.argmax(vals))
    ox = 14 if p_max <= 1 else (-14 if p_max >= len(x_idx) - 2 else 0)
    ax.annotate(f"{etq_mes[p_max].capitalize()}: {fmt_m(vals[p_max])}", xy=(p_max, vals[p_max]),
                xytext=(ox, 20 if i == 0 else 18), textcoords="offset points",
                ha="left" if ox > 0 else ("right" if ox < 0 else "center"), fontsize=9.5, fontweight="bold", color=c,
                bbox=dict(boxstyle="round,pad=0.22", fc=CARD, ec=c, lw=0.9),
                arrowprops=dict(arrowstyle="->", color=c, lw=1.2, shrinkB=4))
ax.set_xticks(x_idx)
ax.set_xticklabels(etq_mes)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: fmt_m(x)))
ax.set_ylim(0, ing_mes_ciudad.values.max() * 1.30)
ax.set(xlabel="Mes", ylabel="Ingresos mensuales")
ax.legend(frameon=False, loc="upper left", ncol=2, fontsize=12)
tot_c = ing_mes_ciudad.sum()
t01 = f"{tot_c.idxmax()} supera a {tot_c.idxmin()} por {fmt_m(tot_c.max() - tot_c.min())}" if len(tot_c) > 1 else f"{tot_c.idxmax()} alcanza {fmt_m(tot_c.max())}"
estilo(ax, t01, "y")

# [0, 2] Dona de canales
ax = axs[0, 2]
ax.set_facecolor(CARD)
_, _, autotexts = ax.pie(
    ing_canal.values, labels=ing_canal.index, autopct="%.1f%%", startangle=90, counterclock=False,
    colors=[VERDE] + [GRISES[i % len(GRISES)] for i in range(len(ing_canal) - 1)],
    explode=[0.035] + [0.0] * (len(ing_canal) - 1), pctdistance=0.73, labeldistance=1.10,
    wedgeprops=dict(width=0.46, edgecolor=CARD, lw=2.2), textprops=dict(fontsize=11.5, color=DARK)
)
for i, at in enumerate(autotexts):
    at.set(color="white" if i == 0 else DARK, fontweight="bold", fontsize=10.5)
ax.text(0, 0.05, "Total 2025", ha="center", va="center", fontsize=10, color=MUTED)
ax.text(0, -0.11, fmt_m(ing_canal.sum()), ha="center", va="center", fontsize=12, fontweight="bold", color=DARK)
ax.set_title(f"Canal {ing_canal.index[0]} concentra el {ing_canal.iloc[0] / ing_canal.sum() * 100:.1f}% de las ventas",
             fontsize=13, fontweight="bold", loc="left", color=DARK, pad=12)

# [1, 0] Barras apiladas categoría por segmento
ax = axs[1, 0]
cats = tabla_seg_cat.columns.tolist()
cat_top = tabla_seg_cat.mean().idxmax()
bottom = np.zeros(len(tabla_seg_cat))
x_seg = np.arange(len(tabla_seg_cat))
g_idx = 0
for cat in cats:
    es_top = cat == cat_top
    color = VERDE if es_top else GRISES[g_idx % len(GRISES)]
    g_idx += not es_top
    vals = tabla_seg_cat[cat].values
    ax.bar(x_seg, vals, bottom=bottom, width=0.55, color=color, label=cat, edgecolor=CARD, lw=1.2)
    for j, v in enumerate(vals):
        if v >= 10:
            ax.text(x_seg[j], bottom[j] + v / 2, f"{v:.1f}%", ha="center", va="center",
                    fontsize=9.5, fontweight="bold", color="white" if es_top else DARK)
    bottom += vals
ax.set_xticks(x_seg)
ax.set_xticklabels(tabla_seg_cat.index)
ax.set_ylim(0, 128)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:.0f}%"
))  # fixed closing brace issue programmatically
ax.set(xlabel="Segmento", ylabel="Participación en ventas (%)")
ax.legend(title="Categoría", ncol=min(3, len(cats)), loc="upper center", frameon=False, fontsize=11.5, title_fontsize=12)
seg_dom, cat_dom = tabla_seg_cat.stack().idxmax()
estilo(ax, f"{cat_dom} representa {tabla_seg_cat.loc[seg_dom, cat_dom]:.1f}% en segmento {seg_dom}", "y")

# [1, 1] Dispersión de vendedores
ax = axs[1, 1]
ciudades_ord = [c for c in COLOR_CIUDAD if c in vend["ciudad"].unique()] + [c for c in vend["ciudad"].unique() if c not in COLOR_CIUDAD]
for ciudad in ciudades_ord:
    d = vend[vend["ciudad"] == ciudad]
    ax.scatter(d["n_ventas"], d["ticket_prom"], label=ciudad, color=COLOR_CIUDAD.get(ciudad, "#64748B"),
               s=85, alpha=0.88, edgecolors="white", lw=0.8)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(fmt_t))
ax.set_ylim(0, vend["ticket_prom"].max() * 1.28)
ax.set(xlabel="Número de ventas por vendedor", ylabel="Ticket promedio")
ax.legend(title="Ciudad", frameon=False, loc="upper right", ncol=2, fontsize=11, title_fontsize=11.5, markerscale=1.25)
t_ciudad = vend.groupby("ciudad")["ticket_prom"].mean()
estilo(ax, f"{t_ciudad.idxmax()} logra el mayor ticket medio ({fmt_t(t_ciudad.max())})", "both")

# [1, 2] Boxplot 2.º semestre
ax = axs[1, 2]
datos_box = [sem2[sem2["segmento"] == s]["total"].dropna().values for s in segmentos_orden]
bp = ax.boxplot(datos_box, labels=segmentos_orden, showfliers=False, patch_artist=True, widths=0.45,
                medianprops=dict(color=DARK, lw=2.2), boxprops=dict(facecolor="#E2E8F0", edgecolor=MUTED, lw=1.1),
                whiskerprops=dict(color=MUTED, lw=1.1), capprops=dict(color=MUTED, lw=1.1))
meds = sem2.groupby("segmento")["total"].median()
seg_top = meds.idxmax()
for i, s in enumerate(segmentos_orden):
    if s == seg_top:
        bp["boxes"][i].set(facecolor=VERDE, alpha=0.88)
        bp["medians"][i].set_color("white")
    q3 = np.percentile(datos_box[i], 75) if len(datos_box[i]) else meds[s]
    ax.text(i + 1, q3 * 1.04, f"Med: {fmt_t(meds[s])}", ha="center", fontsize=9.5,
            fontweight="bold" if s == seg_top else "normal", color=DARK)
max_w = max((np.percentile(d, 75) + 1.5 * (np.percentile(d, 75) - np.percentile(d, 25))) for d in datos_box if len(d))
ax.set_ylim(0, max_w * 1.20)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(fmt_t))
ax.set(xlabel="Segmento", ylabel="Valor por venta (2.º sem.)")
estilo(ax, f"Segmento {seg_top} lidera la mediana semestral ({fmt_t(meds.max())})", "y")

fig.suptitle(f"Desempeño comercial de la cadena de papelerías · 2025 (Total: {fmt_m(df_2025["total"].sum())})",
             fontsize=20, fontweight="bold", color=DARK, x=0.04, y=0.97, ha="left")
plt.tight_layout(rect=[0.02, 0.02, 0.99, 0.92], h_pad=2.8, w_pad=2.4)
"""

with open("figura_v2.py", "w", encoding="utf-8") as f:
    f.write(codigo_v2)

print("¡El archivo figura_v2.py ha sido creado con éxito!")


### Tu dashboard final
Ejecuta la celda: muestra la versión que elegiste y tus tres hallazgos. **Esta es la salida que se verá en GitHub.**

In [ ]:
mostrar(figura_elegida)

print("Hallazgos:")
for h in hallazgos:
    print("•", h)

### Ejercicio 12 · Publica tu notebook en GitHub
GitHub muestra los notebooks **con sus figuras**, así que tu notebook publicado es tu dashboard. Solo necesitas una cuenta gratuita en [github.com](https://github.com).

**Antes de publicar:**
- Ejecuta todo el notebook de arriba abajo (*Entorno de ejecución → Ejecutar todo*) para que las figuras queden guardadas en las salidas.
- **Nunca escribas tu API key dentro de una celda.** Los *Secrets* de Colab no se guardan en el notebook y por eso son seguros; el repositorio será público.

**Pasos:**
1. **Crea el repositorio:** botón **New** → nombre `mi-dashboard-ventas` → **Public** → *Create repository*.
2. **Guarda el notebook en GitHub.** Opción A: en Colab, *Archivo → Guardar una copia en GitHub*, autoriza, elige tu repositorio y acepta. Opción B: *Archivo → Descargar → .ipynb* y, en el repositorio, **Add file → Upload files** y arrastra el archivo.
3. **Abre el notebook en el repositorio** (haz clic en el archivo `.ipynb`): GitHub lo dibuja con tus figuras.
4. **Copia el enlace** de la barra del navegador. Debe verse así: `https://github.com/TU-USUARIO/mi-dashboard-ventas/blob/main/NOMBRE.ipynb`
5. **Compruébalo:** ábrelo en una ventana de incógnito. Si lo ves sin iniciar sesión, está público.

**Si no se ven las figuras:** vuelve a Colab, ejecuta todo y guarda de nuevo. Si GitHub dice *Unable to render code block*, recarga la página unos segundos después.

---
**Para entregar:** (1) el **enlace público** de tu notebook en GitHub, con tu dashboard final visible, todos los integrantes deben estar como colaboradores en el repositorio, y (2) tus prompts (`prompt_v1`, `prompt_v2` y, si la hiciste, `prompt_v3`) más **tres líneas** sobre qué tuviste que corregir de lo que hizo Gemini y cómo lo pediste (escríbelas en una celda de texto al final del notebook) (3) El notebook (ipynb) con los nombres de todos los integrantes.